In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [14]:
from utils import *
import os
import torch
import tensorflow as tf
from cnn.model import ConvBlock
import cnn_tf.model as tf_model
import numpy as np

In [7]:
# disable all tensorflow logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [3]:
# Load config file
file_path = os.path.join(os.getcwd(), 'config_files', 'config2.txt')
config_file = load_yaml(file_path)

In [5]:
kernel = 3
stride = 1
filters = 32
mu=0.9
epsilon=2e-5
conv_block = ConvBlock(kernel = kernel,
                       strides = stride, 
                       filters = filters, 
                       mu=mu,
                       epsilon=epsilon)

In [6]:
tf_conv_block = tf_model.ConvBlock(kernel = kernel,
                                   strides = stride,
                                   filters = filters,
                                   mu=mu, 
                                   epsilon=epsilon)

In [25]:
tf.random.set_seed(42)
torch.manual_seed(42)

x_torch = torch.randn(1,6,6,3)
x_tf = tf.random.normal((1,6,6,3))

y_torch = conv_block(x_torch).detach().numpy()
y_tf = tf_conv_block(x_tf, name="conv_block")

print(f"torch: {y_torch.shape}, tensorflow: {y_tf.shape}")

torch: (1, 6, 6, 32), tensorflow: (1, 6, 6, 32)


In [29]:
np.isclose(y_torch, y_tf).all()

False

In [31]:
from cnn.model import MaxPooling

In [34]:
# create a a image tensor to test the max pooling layer
max_pool = MaxPooling(kernel=3, strides=1)
tf_max_pool = tf_model.MaxPooling(kernel=3, strides=1)

y_tf = tf_max_pool(x_tf, name="max_pool")
y_torch = max_pool(x_torch).detach().numpy()

print(f"torch: {y_torch.shape}, tensorflow: {y_tf.shape}")

torch: (1, 4, 4, 3), tensorflow: (1, 4, 4, 3)


In [36]:
from cnn.model import AvgPooling

avg_pool = AvgPooling(kernel=3, strides=1)
y_torch = avg_pool(x_torch).detach().numpy()

tf_avg_pool = tf_model.AvgPooling(kernel=3, strides=1)
y_tf = tf_avg_pool(x_tf, name='avg_pool')

print(f"torch: {y_torch.shape}, tensorflow: {y_tf.shape}")

torch: (1, 4, 4, 3), tensorflow: (1, 4, 4, 3)


In [ ]:
from cnn.model import FullyConnected

x = torch.randn(1,6,6,3)

# flatten the tensor
num_features = x.shape[1] * x.shape[2] * x.shape[3]

fc = FullyConnected(inputs_features=num_features, units=10)

x_t = torch.reshape(x, [-1, num_features])
y = fc(x_t)
print(f"Shape of x: {x_t.shape} and shape of y: {y.shape}")

In [ ]:
from cnn.model import pad_features

tensors = [torch.randn(1, 32, 32,3), torch.randn(1, 32, 32,6)]
tensor_padded = pad_features(tensors=tensors)
tensor_padded[0].shape, tensor_padded[1].shape